# 先认识 Jev：从聊天文本到软件决策（AI Primer Lab）

针对官方文档对应章节的可运行实验笔记，全部实验使用**中文场景与中文提示词**。
面向会基础 Python、刚接触 AI Agent 的读者。

**学习目标：** 先说明 Jev 与 System One 的关系，再用一次赛事调度实验连接类型化答案、概率和代码控制流。

[官方原文](https://docs.typesafe.ai/introduction/machine-learning-primer) · [中文参考](https://bald0wang.github.io/jev-docs-zh/introduction/machine-learning-primer/)。本章以中文重述理论、复刻对应场景；扩展实验会单独说明。
所有客户、订单及消息均为教学合成数据。

## 笔记本结构

| 章节 | 内容 |
|---|---|
| 0. 准备 | 安装库、配置客户端、连通性测试与离线示例 |
| 1 | Jev 是什么：Introduction 与 AI Primer 的主张 |
| 2 | 赛事指挥台：Choice、Score、Noul 如何组成一次分流 |
| 3 | 概率属于一组预测：用小数据读懂校准 |
| 练习与小结 | 练习、自查、总结与本次执行记录 |

实验按**原理 → 理论根基 → 定义数据 → 定义问题 → 调用 → 解读结果**展开，每个代码单元格只做一件事。

## 运行要求

- Python ≥ 3.10；本章使用 `typesafe-sdk==0.7.0`。
- 真实实验需要启动进程的 `TYPESAFE_API_KEY` 环境变量，密钥不要写进 Notebook。

在本仓库 `notebooks/` 目录创建环境并打开本文件：

```bash
./setup_env.sh
.venv/bin/python -m pip install -r requirements.txt -c generators/constraints-foundations.txt
.venv/bin/jupyter lab ai_primer_experiments.ipynb
```

产品名、字段名和选项 key 保持英文，state、提示词与解说使用中文。
默认 `JEV_RUN_MODE=live`，调用失败即停止；无密钥学习时，在启动 Jupyter 前设置 `JEV_RUN_MODE=offline`。
`auto` 仅供教学体验，缺密钥或 401 时显式回退；正式验收使用 `live`。

**验证状态：真实 API 待验收。** 本文件尚未执行真实 API；离线检查仅验证代码路径。
批量执行、离线预览和验收记录见本目录 `MAINTENANCE.md`。

## 0. 准备

本节可折叠阅读，但独立运行时不能跳过。客户端、辅助对象和示例数据都在本文件中定义。

### 0.1 安装依赖

推荐先运行 `setup_env.sh`。只有当前内核缺少 SDK 时，本格才安装依赖。

In [1]:
import importlib.util
if importlib.util.find_spec("typesafe_sdk") is None:
    %pip install -q typesafe-sdk==0.7.0

**观察与理解：** 安装包的名字是 typesafe-sdk，Python 导入名是 typesafe_sdk。安装成功不代表 API 已连通。

### 0.2 导入与配置

默认模型固定版本，便于记录实验条件；可通过环境变量更换。不要从 Notebook 输入密钥。

In [2]:
import os
import json
import time
import math
from datetime import datetime, timezone
from importlib.metadata import version
from typesafe_sdk import (
    Choice, Score, Noul, NoulCriteria, TypeSafeClient,
    TypeSafeAuthenticationError, RetryPolicy,
)

MODEL = os.environ.get("TYPESAFE_DEFAULT_MODEL", "jev-1.13.0")
RUN_MODE = os.environ.get("JEV_RUN_MODE", "live")
API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
if RUN_MODE not in {"live", "offline", "auto"}:
    raise ValueError("JEV_RUN_MODE 只能是 live、offline 或 auto")
if RUN_MODE == "live" and not API_KEY:
    raise RuntimeError("请在启动 Jupyter 前配置 TYPESAFE_API_KEY 环境变量")
client = None
if RUN_MODE != "offline" and API_KEY:
    client = TypeSafeClient(api_key=API_KEY, model=MODEL, timeout=30,
                           retry=RetryPolicy(max_retries=0))
print("模式：", RUN_MODE, "SDK：", version("typesafe-sdk"), "模型配置：", MODEL)

模式： auto SDK： 0.7.1 模型配置： jev-1.13.0


正式验收禁用自动回退，且不自动重试，以便请求数量有界。`auto` 与 `offline` 是教学工具，不代表成功连接模型。

### 0.3 连通性测试

用一条 Noul 检查真实响应能否返回。网络、限流与输入错误直接抛出，不伪装成不确定判断。

In [3]:
PING = {"source": "offline", "reason": "未发起连通性请求"}
if client is not None:
    try:
        ping = client.system_one("你好", {"greeting": Noul(
            instructions="这段文字是否在打招呼？")})
        PING = {"source": "live", "model": ping.model,
                "input_tokens": ping.usage.input_tokens,
                "output_tokens": ping.usage.output_tokens}
    except TypeSafeAuthenticationError:
        if RUN_MODE == "live":
            raise
        client.close()
        client = None
        PING["reason"] = "401 鉴权失败，仅教学模式允许回退"
print(json.dumps(PING, ensure_ascii=False))

{"source": "offline", "reason": "未发起连通性请求"}


**观察与理解：** source=live 表示这一次连通性请求成功；仍要查看后续实验记录，不能用它代替整章验收。

### 0.4 离线替身

沿用参考模板的 `_FakeAnswer` 与 `_FakeResponse` 访问方式。人工数字仅用来检验读取字段和代码分支。

In [4]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for name, value in values.items():
            setattr(self, name, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.model = "人工示例，非模型预测"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)

人工 Score 由概率计算期望，避免模板中的分数与分布不一致。人工 confidence 只是指定的演示字段，不是在复现服务端的计算公式。

定义两种示例答案构造器；Noul 可直接用 `_FakeAnswer`。所有具体答案集中在下一节。

In [5]:
def fake_choice(probabilities, confidence):
    return _FakeAnswer("choice", choice=max(probabilities, key=probabilities.get),
                       probabilities=probabilities, confidence=confidence)


def fake_score(probabilities, legend, confidence):
    return _FakeAnswer("score", score=sum(k * p for k, p in probabilities.items()),
                       probabilities=probabilities, legend=dict(enumerate(legend)),
                       confidence=confidence)

**观察与理解：** 例如概率 {0:0.05, 1:0.26, 2:0.69} 对应 1.64。不能把另一个数与该分布配在一起。

### 0.5 统一调用入口

每次调用记录来源、模型与 token 用量。离线耗时记为 None，不把本地字典访问当成模型速度。

In [6]:
CALL_LOG = []


class TS:
    def call(self, state, questions, offline_answers, label):
        start = time.perf_counter()
        source = "live"
        if client is None:
            response, source = _FakeResponse(offline_answers), "offline"
        else:
            try:
                response = client.system_one(state, questions)
            except TypeSafeAuthenticationError:
                if RUN_MODE != "auto":
                    raise
                response, source = _FakeResponse(offline_answers), "offline"
        validate_response(response, questions)
        CALL_LOG.append({"case": label, "source": source, "model": response.model,
                         "seconds": time.perf_counter() - start if source == "live" else None,
                         "input_tokens": response.usage.input_tokens,
                         "output_tokens": response.usage.output_tokens})
        if source == "offline":
            print("离线示例：", label, "；人工答案，不是 Jev 实测")
        return response


ts = TS()

与参考模板相比，这里增加了严格 live 模式和逐次记录。保留 401 教学回退，但超时、429 等错误继续失败，防止验收被回退掩盖。

校验结构与数值契约；只断言接口应满足的性质，不断言真实模型必须预测某个标签。

In [7]:
def validate_response(response, questions):
    if set(response.answers) != set(questions):
        raise ValueError("答案 ID 与问题 ID 不一致")
    for key, question in questions.items():
        answer = response.answers[key]
        if isinstance(question, Noul):
            if not 0 <= answer.noul <= 1:
                raise ValueError("Noul 超出概率范围")
            continue
        probabilities = answer.probabilities
        if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities.values()):
            raise ValueError("概率值无效")
        if not math.isclose(sum(probabilities.values()), 1, abs_tol=0.02):
            raise ValueError("概率之和偏离 1")
        if not 0 <= answer.confidence <= 1:
            raise ValueError("confidence 超出范围")
        if isinstance(question, Choice):
            if set(probabilities) != set(question.criteria):
                raise ValueError("Choice 选项集合不一致")
            if answer.choice not in probabilities:
                raise ValueError("Choice 标签不在选项中")
        else:
            expected = sum(int(k) * p for k, p in probabilities.items())
            if not math.isclose(answer.score, expected, abs_tol=0.03):
                raise ValueError("Score 与概率加权期望不一致")

**观察与理解：** 容差用于服务端数值舍入。结构检查通过只说明响应可读取，不证明语义判断正确。

显示结果时统一列出类型、概率和置信度；Noul 不额外制造 confidence 字段。

In [8]:
def show(response):
    rows = {}
    for key, answer in response.answers.items():
        rows[key] = {name: getattr(answer, name) for name in
                     ("type", "choice", "score", "noul", "confidence", "probabilities", "legend")
                     if hasattr(answer, name)}
    print(json.dumps(rows, ensure_ascii=False, indent=2))

### 0.6 本章离线示例数据

以下数值全部人工构造，专门测试分支；不来自 Jev，也不能用于估计中文准确率或校准情况。正式 live 运行不会使用这些答案。

In [9]:
PRIORITY_LEVELS = [
    "可在赛事结束后处理",
    "尽量在 30 分钟内处理",
    "需要在 10 分钟内处理",
    "必须立即处理",
]

In [10]:
INCIDENTS = [
    {"id": "runner_injury", "state": {
        "event": "海湾夜跑 10 公里",
        "time": "19:12",
        "message": "2.4 公里蓝旗处有跑者脚踝扭伤，无法继续前进，请派人协助。",
    }},
    {"id": "water_station", "state": {
        "event": "海湾夜跑 10 公里",
        "time": "19:12",
        "message": "3 号补水点只剩两箱水，下一批跑者约两分钟后到。",
    }},
    {"id": "shirt_pickup", "state": {
        "event": "海湾夜跑 10 公里",
        "time": "19:12",
        "message": "完赛纪念衫在哪个窗口领取？我刚到终点。",
    }},
]

In [11]:
INCIDENT_OFFLINE = [
    {
        "incident_type": fake_choice({"medical_help": .92, "resource_shortage": .03,
                                       "event_information": .03, "other": .02}, .92),
        "urgency": fake_score({0: 0, 1: 0, 2: .2, 3: .8}, PRIORITY_LEVELS, .82),
        "requires_human": _FakeAnswer("noul", noul=.98),
    },
    {
        "incident_type": fake_choice({"medical_help": .02, "resource_shortage": .84,
                                       "event_information": .10, "other": .04}, .84),
        "urgency": fake_score({0: 0, 1: .1, 2: .8, 3: .1}, PRIORITY_LEVELS, .73),
        "requires_human": _FakeAnswer("noul", noul=.74),
    },
    {
        "incident_type": fake_choice({"medical_help": .01, "resource_shortage": .01,
                                       "event_information": .94, "other": .04}, .94),
        "urgency": fake_score({0: .8, 1: .15, 2: .05, 3: 0}, PRIORITY_LEVELS, .82),
        "requires_human": _FakeAnswer("noul", noul=.04),
    },
]

## 1. 先回答一个问题：Jev 是什么？

官方 **Introduction** 把 Jev 定义为 TypeSafe 的旗舰模型，也是第一个 **System One** 模型。TypeSafe 是平台；Jev 是模型；System One 描述的是一类面向软件、快速回答窄问题的模型。

一次请求把业务材料放进 `state`，再用 `Choice`、`Score` 或 `Noul` 写出要判断的问题。Jev 返回有类型的答案和概率，Python 直接读取字段并决定下一步；它不先写一段给人看的文字，再让程序猜其中的 JSON。

这也不是“让模型接管整个应用”。下面由 Jev 判断赛事消息，阈值、人工转交和队列分派仍由普通代码控制。Notebook 只打印模拟分流，不会联系工作人员或执行现场动作。

阅读：[官方 Introduction](https://docs.typesafe.ai/introduction) · [本地知识库 Introduction](../dist/introduction/index.html) · {sources("introduction")}

## 2. AI Primer 补充了什么？

AI Primer 解释 TypeSafe 为什么把模型训练目标放在**可校准的决策概率**上。文档用 RLHF（偏好回答）、RLVR（可验证奖励）和 RLCD（面向校准决策的强化学习）对照不同目标，并将机器可读、可测试的输出称为 Machine Native Intelligence。

这是 TypeSafe 对自身方向的说明，不是对所有聊天模型的概括，也没有公开足以复现 RLCD 的完整训练配方。这里先看 Jev 的输入/输出契约；不会从训练目标直接推导某个业务任务一定准确。

![官方图示：预训练模型之后的 RLHF、RLVR 与 RLCD 路径](../assets/images/images/ai-primer/training-paths-light.webp)

校准也不是“这一次有 80% 把握就必然正确”。它描述很多次预测组成的群体：被赋予 0.8 概率的事件，长期发生频率应大致接近 80%。下方实验把这个定义变成能运行的分流流程。

阅读：[官方 AI Primer](https://docs.typesafe.ai/introduction/machine-learning-primer) · [本地知识库 AI Primer](../dist/introduction/machine-learning-primer/index.html) · {sources("introduction/machine-learning-primer")} · {sources("confidence")}

## 3. 实验：海湾夜跑的赛事指挥台

三条现场消息都放进相同形状的 `state`。我们让 Jev 在一次请求中各回答三个**互相独立**的问题：属于哪类事件、需要多快处理、是否需要人工介入。之后由 Python 组合结果。

| 问题 | 原语 | 代码将怎样使用 |
|---|---|---|
| 事件类别 | `Choice` | 选现场医疗、补给、赛事信息或其他 |
| 处理时限 | `Score` | 按 0–3 的有序等级安排优先级 |
| 是否要人工介入 | `Noul` | 读取命题概率；高风险直接交给现场负责人 |

**接口边界：** `Choice` 和 `Score` 返回 `confidence` 与概率分布；`Noul` 返回 `noul` 概率，没有独立的 `confidence` 字段。阈值是应用政策，正式使用前需要独立标注数据来选择和验证。

### 3.1 写出原子问题

每题只判断一个维度；问题描述写清标准，不能指望 ID 名称代替 instructions。

In [12]:
INCIDENT_QUESTIONS = {
    "incident_type": Choice(
        instructions="这条赛事消息主要属于哪类事件？",
        criteria={
            "medical_help": "跑者受伤或需要现场医疗协助",
            "resource_shortage": "补水、物资或现场设施问题",
            "event_information": "路线、时间或领取点等赛事信息问题",
            "other": "其他情况或现有信息无法归类",
        },
    ),
    "urgency": Score(
        instructions="仅根据消息判断处理时限；不要把类别本身当作紧急程度。",
        criteria=PRIORITY_LEVELS,
    ),
    "requires_human": Noul(
        instructions="在自动处理前，这条消息是否需要现场人员核实或介入？",
    ),
}

**观察与理解：** 三个问题共享一份 state，但答案通过各自的 ID 取回；一次调用不会让某题先读到另一题答案。

### 3.2 一次请求回答三题

每条消息各发一次请求；总计 3 次实验请求，另有准备区的 1 次连通性请求。

In [13]:
incident_responses = [
    ts.call(incident["state"], INCIDENT_QUESTIONS, offline, incident["id"])
    for incident, offline in zip(INCIDENTS, INCIDENT_OFFLINE)
]

离线示例： runner_injury ；人工答案，不是 Jev 实测
离线示例： water_station ；人工答案，不是 Jev 实测
离线示例： shirt_pickup ；人工答案，不是 Jev 实测


**观察与理解：** 离线模式使用上面的人工答案，只检验流程。Live 模式才会请求 Jev；不同措辞或模型版本可能给出不同结果。

### 3.3 看模型返回的类型化答案

逐条查看答案类型、概率和 confidence；不要只看最终队列。

In [14]:
for incident, response in zip(INCIDENTS, incident_responses):
    print("事件:", incident["id"])
    show(response)
    print()

事件: runner_injury
{
  "incident_type": {
    "type": "choice",
    "choice": "medical_help",
    "confidence": 0.92,
    "probabilities": {
      "medical_help": 0.92,
      "resource_shortage": 0.03,
      "event_information": 0.03,
      "other": 0.02
    }
  },
  "urgency": {
    "type": "score",
    "score": 2.8000000000000003,
    "confidence": 0.82,
    "probabilities": {
      "0": 0,
      "1": 0,
      "2": 0.2,
      "3": 0.8
    },
    "legend": {
      "0": "可在赛事结束后处理",
      "1": "尽量在 30 分钟内处理",
      "2": "需要在 10 分钟内处理",
      "3": "必须立即处理"
    }
  },
  "requires_human": {
    "type": "noul",
    "noul": 0.98
  }
}

事件: water_station
{
  "incident_type": {
    "type": "choice",
    "choice": "resource_shortage",
    "confidence": 0.84,
    "probabilities": {
      "medical_help": 0.02,
      "resource_shortage": 0.84,
      "event_information": 0.1,
      "other": 0.04
    }
  },
  "urgency": {
    "type": "score",
    "score": 2.0,
    "confidence": 0.73,
    "probabilit

### 3.4 用代码决定是否自动分流

In [15]:
HUMAN_REVIEW_THRESHOLD = 0.75

def dispatch(response, human_threshold=HUMAN_REVIEW_THRESHOLD):
    incident = response.choices["incident_type"]
    urgency = response.scores["urgency"]
    requires_human = response.nouls["requires_human"].noul
    if incident.choice == "medical_help" or requires_human >= human_threshold:
        return "现场负责人 / 医疗人员"
    if incident.confidence < 0.60 or urgency.confidence < 0.60:
        return "值班台补充信息"
    if incident.choice == "resource_shortage" and urgency.score >= 1.5:
        return "补给组（优先处理）"
    if incident.choice == "event_information" and urgency.score < 1.0:
        return "自动回复赛事 FAQ（模拟）"
    return "普通运维队列"

**观察与理解：** 医疗类总是转给现场人员；其他类先检查人工介入概率和 confidence，再由代码应用各自的处理规则。

输出指挥台结果。分流只是本地字符串，不会真的派工。

In [16]:
routes = [dispatch(response) for response in incident_responses]
for incident, response, route in zip(INCIDENTS, incident_responses, routes):
    print(f"{incident['id']:<16} → {route}")

observed_live = {
    route for route, log in zip(routes, CALL_LOG) if log["source"] == "live"
}
COVERAGE = {
    "live_routes_observed": sorted(observed_live),
    "offline_routes_are_model_evidence": False,
}

runner_injury    → 现场负责人 / 医疗人员
water_station    → 补给组（优先处理）
shirt_pickup     → 自动回复赛事 FAQ（模拟）


**观察与理解：** 若 live 返回没有命中某条路径，把它记为本次未观察到；不要改写模型答案来凑齐分支。

### 3.5 同一答案，改一条政策阈值

In [17]:
resource_response = incident_responses[1]
for threshold in (0.70, 0.75):
    print({
        "人工介入阈值": threshold,
        "结果": dispatch(resource_response, human_threshold=threshold),
    })

{'人工介入阈值': 0.7, '结果': '现场负责人 / 医疗人员'}
{'人工介入阈值': 0.75, '结果': '补给组（优先处理）'}


**观察与理解：** 这里复用同一组答案，只改 Python 阈值。若路由改变，改变的是应用政策，不是 Jev 的输出。0.70/0.75 只是演示值，不是生产阈值。

## 4. 一张小表读懂校准

这组数据是人工构造的数学演示，不是 Jev 结果。每个概率桶各有 10 条记录，刻意让实际频率与预测概率相等。

构造两个概率桶。1 表示事件发生，0 表示没有发生。

In [18]:
CALIBRATION_GROUPS = {
    0.2: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    0.8: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
}

比较预测概率与每组实际发生率。

In [19]:
calibration_table = [
    {"预测概率": probability, "样本数": len(labels),
     "实际频率": sum(labels) / len(labels)}
    for probability, labels in CALIBRATION_GROUPS.items()
]

打印结果，并数一数 0.8 组里有几次没发生。

In [20]:
print(json.dumps(calibration_table, ensure_ascii=False, indent=2))
print("0.8 概率组中的未发生次数:", CALIBRATION_GROUPS[0.8].count(0))

[
  {
    "预测概率": 0.2,
    "样本数": 10,
    "实际频率": 0.1
  },
  {
    "预测概率": 0.8,
    "样本数": 10,
    "实际频率": 0.8
  }
]
0.8 概率组中的未发生次数: 2


**观察与理解：** 这组 10 条中仍有 2 条未发生。校准描述群体频率，不会保证某一条预测正确；10 条也太少，不能据此评估真实模型。

## 练习与自查

如果主办方决定：任何医疗消息都不能自动关闭工单，你会把这条规则放进问题措辞，还是放进 Python 控制流？为什么？

<details><summary>参考思路：先完成练习再展开</summary>

应把不可妥协的安全规则写进确定性控制流；Jev 可以帮助识别医疗类消息，代码确保该类始终交给现场人员。之后仍要用有标签的独立数据检查识别漏报。

</details>

## 小结

| 组件 | 在实验中的职责 |
|---|---|
| Jev / System One | 对同一 state 回答窄而明确的 typed questions |
| 概率与 confidence | 暴露不确定性信号；不能替代业务验证 |
| Python | 设置阈值、组合结果、控制分支和人工出口 |

下一步：[快速开始实验](quickstart_experiments.ipynb)带你亲手发出一次请求；之后可看[简介实验](introduction_experiments.ipynb)练习组合判断。训练目标与实现细节以官方 [AI Primer](https://docs.typesafe.ai/introduction/machine-learning-primer) 为准。

离线运行只说明教材代码能执行。正式交付必须实际运行 live，并阅读每条输出；缺失的分支应记为未观察到。

## 本次执行记录

先关闭连接，再生成记录。下面的 JSON 由实际运行计算，批量执行器会据此检查来源。

In [21]:
if client is not None:
    client.close()

真实探针只演示行为路径；若据其返回挑选样例，这批样例就不适合再当作无偏准确率测试集。延迟也只是本次网络环境中的观测。

In [22]:
AUDIT = {
    "kind": "jev_execution_audit",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "sdk": version("typesafe-sdk"), "requested_model": MODEL,
    "mode": RUN_MODE, "ping": PING,
    "real_calls": sum(x["source"] == "live" for x in CALL_LOG),
    "offline_calls": sum(x["source"] == "offline" for x in CALL_LOG),
    "cases": CALL_LOG,
    "coverage": globals().get("COVERAGE", {}),
    "validation_status": "live_executed_requires_review" if (
        PING["source"] == "live" and CALL_LOG
        and all(x["source"] == "live" for x in CALL_LOG)
    ) else "offline_only_not_model_evidence",
}
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))

{
  "kind": "jev_execution_audit",
  "executed_at_utc": "2026-09-25T05:49:25.821736+00:00",
  "sdk": "0.7.1",
  "requested_model": "jev-1.13.0",
  "mode": "auto",
  "ping": {
    "source": "offline",
    "reason": "未发起连通性请求"
  },
  "real_calls": 0,
  "offline_calls": 3,
  "cases": [
    {
      "case": "runner_injury",
      "source": "offline",
      "model": "人工示例，非模型预测",
      "seconds": null,
      "input_tokens": 0,
      "output_tokens": 0
    },
    {
      "case": "water_station",
      "source": "offline",
      "model": "人工示例，非模型预测",
      "seconds": null,
      "input_tokens": 0,
      "output_tokens": 0
    },
    {
      "case": "shirt_pickup",
      "source": "offline",
      "model": "人工示例，非模型预测",
      "seconds": null,
      "input_tokens": 0,
      "output_tokens": 0
    }
  ],
  "coverage": {
    "live_routes_observed": [],
    "offline_routes_are_model_evidence": false
  },
  "validation_status": "offline_only_not_model_evidence"
}


读完输出后，在本仓库 `notebooks/MAINTENANCE.md` 的验收表中记录日期、真实模型、观察到的分支和偏离预期之处。不要把人工演示数值抄进实测记录。